# CEKAS · Catatan Keuangan Kelas
Kelas XI-F2 SMA Kartika XIX-1 Bandung · Kelompok 5

> **Notebook ini hanya etalase kode.** Sistem utama berjalan 24/7 di Google Cloud Run (bot Telegram + dashboard web), bukan di Colab.

- Dashboard publik: https://cekas-pgku264xda-uc.a.run.app
- Kode: https://github.com/Ardelyo/cekas-kas-kelas

**Fitur utama:** buku kas append-only dengan hash chain, 4 pos anggaran, iuran mingguan, koreksi (bukan hapus), verifikasi Wali Kelas, usulan belanja yang divoting, rekap WhatsApp, laporan SPJ, replikasi SQLite ke Cloud Storage.

## 1. Ambil kode dari GitHub

In [ ]:
!rm -rf cekas-kas-kelas
!git clone -q https://github.com/Ardelyo/cekas-kas-kelas.git
%cd cekas-kas-kelas
!pip install -q -r requirements.txt
!ls

## 2. Demo end-to-end (database sementara di memori, bukan data kelas asli)

In [ ]:
!python cekas_demo.py --demo

## 3. Tes otomatis (termasuk alur verifikasi Wali Kelas)

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -6

## 4. Integritas buku kas: ubah satu baris, rantai hash ketahuan

In [ ]:
from cekas_app.database import CekasDB
from cekas_app.config import CLASS_ID
db = CekasDB(":memory:"); db.get_or_create_class(CLASS_ID)
db.record_transaction(CLASS_ID, "in", 50000, "operasional", "Iuran kas", "Demo", 1)
db.record_transaction(CLASS_ID, "out", 20000, "operasional", "Beli spidol", "Demo", 1)
print("Sebelum diubah:", db.verify_ledger(CLASS_ID))
db.conn.execute("UPDATE transactions SET amount = 5 WHERE id = (SELECT MIN(id) FROM transactions)")
db.conn.commit()
print("Setelah satu baris diubah:", db.verify_ledger(CLASS_ID))

## 5. Peta kode
```
cekas_app/database.py   SQLite WAL, ledger hash chain, peran, verifikasi guru
cekas_app/bot.py        bot Telegram (webhook)
cekas_app/web.py        dashboard + login Bendahara
cekas_app/gcs_store.py  replikasi ke Cloud Storage
deploy_gcp.py           deploy Cloud Run satu perintah
```